# 01 — Data Ingestion & Schema Check

**Phase 1, Week 1** — Load the public Kaggle Smart Meter Electricity Consumption Dataset, parse timestamps, and verify schema and 30-minute time-series continuity.

**Scope:** Ingestion and validation only. No EDA, visualizations, or modeling.

In [ ]:
from pathlib import Path

# --- Choose ONE runtime setup below (uncomment the block you need) ---

# A) Google Colab — mount Drive and point to your cloned repo
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = Path("/content/drive/MyDrive/energy-anomaly-forecasting")

# B) Kaggle — download dataset via Kaggle API
# Place kaggle.json in ~/.kaggle/ or use Kaggle notebook secrets.
# !pip install -q kaggle
# !kaggle datasets download -d ziya07/smart-meter-electricity-consumption-dataset -p /tmp/data --unzip
# PROJECT_ROOT = Path("/tmp/data")

# C) Local / cloned repo (default)
PROJECT_ROOT = Path("..").resolve()

In [ ]:
# Canonical source: src/data/ingest_data.py — keep logic in sync when updating.

from pathlib import Path

import pandas as pd

DATASET_FOLDER_NAME = "Smart Meter Electricity Consumption Dataset"
EXPECTED_FILENAME = "smart_meter_data.csv"


def find_dataset_csv(root: Path) -> Path:
    candidates: list[Path] = []

    raw_dir = root / "data" / "raw"
    if raw_dir.is_dir():
        candidates.extend(sorted(raw_dir.glob("*.csv")))

    dataset_dir = root / DATASET_FOLDER_NAME
    if dataset_dir.is_dir():
        candidates.extend(sorted(dataset_dir.glob("*.csv")))

    candidates.extend(sorted(root.glob(f"**/{EXPECTED_FILENAME}")))

    seen: set[Path] = set()
    unique_candidates: list[Path] = []
    for path in candidates:
        resolved = path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique_candidates.append(resolved)

    if not unique_candidates:
        raise FileNotFoundError(
            f"No CSV found under {root}. Expected data/raw/*.csv, "
            f"{DATASET_FOLDER_NAME}/*.csv, or **/{EXPECTED_FILENAME}"
        )

    return unique_candidates[0]


def load_smart_meter_data(csv_path: Path | str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%Y-%m-%d %H:%M:%S")
    return df.sort_values("Timestamp").reset_index(drop=True)


def print_schema_summary(df: pd.DataFrame) -> None:
    print("=" * 60)
    print("SCHEMA SUMMARY")
    print("=" * 60)
    print(f"Shape: {df.shape}")
    print(f"\nColumns ({len(df.columns)}):")
    for col in df.columns:
        print(f"  - {col}")
    print("\nData types:")
    print(df.dtypes.to_string())
    print("\nNull counts:")
    print(df.isna().sum().to_string())
    print("=" * 60)


def check_time_continuity(df: pd.DataFrame, freq: str = "30min") -> None:
    print("=" * 60)
    print("TIME-SERIES CONTINUITY CHECK")
    print("=" * 60)

    ts = df["Timestamp"]
    expected_delta = pd.Timedelta(freq)

    duplicate_count = int(ts.duplicated().sum())
    deltas = ts.diff().dropna()
    irregular_count = int((deltas != expected_delta).sum())

    expected_index = pd.date_range(start=ts.min(), end=ts.max(), freq=freq)
    observed_set = set(ts)
    missing_timestamps = [t for t in expected_index if t not in observed_set]

    print(f"Frequency:           {freq}")
    print(f"Start timestamp:     {ts.min()}")
    print(f"End timestamp:       {ts.max()}")
    print(f"Observed rows:         {len(df)}")
    print(f"Expected rows:         {len(expected_index)}")
    print(f"Missing timestamps:    {len(missing_timestamps)}")
    print(f"Duplicate timestamps:  {duplicate_count}")
    print(f"Irregular intervals:   {irregular_count}")

    if missing_timestamps:
        print("\nFirst missing timestamps (up to 10):")
        for t in missing_timestamps[:10]:
            print(f"  - {t}")

    if duplicate_count == 0 and irregular_count == 0 and not missing_timestamps:
        print("\nResult: PASS — continuous 30-minute series with no gaps or duplicates.")
    else:
        print("\nResult: REVIEW — continuity issues detected (see counts above).")

    print("=" * 60)

In [ ]:
csv_path = find_dataset_csv(PROJECT_ROOT)
df = load_smart_meter_data(csv_path)
print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded CSV:   {csv_path}")

In [ ]:
print_schema_summary(df)

print("\nFirst 3 rows:")
display(df.head(3))

print("\nLast 3 rows:")
display(df.tail(3))

In [ ]:
check_time_continuity(df)

---

**Phase 1 Week 1 complete.** Do not proceed to EDA or modeling in this notebook yet.